In [1]:
!pip install gradio yfinance pandas numpy scikit-learn transformers torch requests beautifulsoup4 matplotlib

In [2]:
import yfinance as yf
import pandas as pd

def fetch_commodity_data(ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
    """
    Fetches historical daily market data for a given commodity ticker.
    """
    print(f"Fetching market data for {ticker}...")
    # Fetch data
    df = yf.download(ticker, start=start_date, end=end_date)

    if df.empty:
        raise ValueError(f"No data found for ticker {ticker}. Check the symbol or dates.")

    # Flatten multi-level columns if present (yfinance sometimes returns them)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    # Reset index to bring Date in as a column
    df = df.reset_index()

    # Clean column names
    df.columns = [col.lower().replace(' ', '_') for col in df.columns]

    # Calculate simple daily price target (Close price shifting by -1 day to predict tomorrow)
    df['target_next_close'] = df['close'].shift(-1)

    # Drop the last row because it won't have a next day target yet
    df = df.dropna(subset=['target_next_close'])

    print(f"Successfully fetched {len(df)} rows of price data.")
    return df

# Quick Test for Gold
# test_df = fetch_commodity_data("GC=F", "2025-01-01", "2026-01-01")
# print(test_df.head(2))

In [3]:
import yfinance as yf
import pandas as pd

def fetch_commodity_data(ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
    """
    Fetches historical daily market data for a given commodity ticker.
    """
    print(f"Fetching market data for {ticker}...")
    # Fetch data
    df = yf.download(ticker, start=start_date, end=end_date)

    if df.empty:
        raise ValueError(f"No data found for ticker {ticker}. Check the symbol or dates.")

    # Flatten multi-level columns if present (yfinance sometimes returns them)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    # Reset index to bring Date in as a column
    df = df.reset_index()

    # Clean column names
    df.columns = [col.lower().replace(' ', '_') for col in df.columns]

    # Calculate simple daily price target (Close price shifting by -1 day to predict tomorrow)
    df['target_next_close'] = df['close'].shift(-1)

    # Drop the last row because it won't have a next day target yet
    df = df.dropna(subset=['target_next_close'])

    print(f"Successfully fetched {len(df)} rows of price data.")
    return df

# Quick Test for Gold
# test_df = fetch_commodity_data("GC=F", "2025-01-01", "2026-01-01")
#  print(test_df.head(2))

In [4]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

# ----------------------------------------------------
# PART 4: DATA PREPROCESSING & FEATURE ENGINEERING
# ----------------------------------------------------

def engineer_features(price_df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculates technical indicators to give the ML model numerical features.
    """
    df = price_df.copy()

    # Ensure date is a datetime object and sort
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)

    # 1. Simple Moving Averages (SMA)
    df['sma_5'] = df['close'].rolling(window=5).mean()
    df['sma_20'] = df['close'].rolling(window=20).mean()

    # 2. Daily Volatility (Standard Deviation of returns over 5 days)
    df['daily_return'] = df['close'].pct_change()
    df['volatility_5'] = df['daily_return'].rolling(window=5).std()

    # Fill any NaNs created by rolling windows with backfill
    df = df.bfill()

    return df




In [5]:
# ----------------------------------------------------
# PART 5: TEXT EMBEDDING & SENTIMENT ANALYSIS (GPU Accelerated)
# ----------------------------------------------------

# Check if T4 GPU is available, otherwise default to CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device} for text embedding.")

# Load a lightweight financial/general transformer model from Hugging Face
# 'distilbert-base-uncased' is fast, free, and gives great semantic vectors
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)

def get_text_embedding(text: str) -> np.ndarray:
    """
    Converts a headline string into a 768-dimensional numerical vector using the GPU.
    """
    if not text.strip():
        return np.zeros(768)

    # Tokenize and push vectors to GPU memory
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    # Take the mean of the hidden states across tokens to get a single vector for the sentence
    embeddings = outputs.last_hidden_state.mean(dim=1)

    # Bring back to CPU and convert to numpy array
    return embeddings.cpu().numpy().flatten()

def aggregate_daily_news(news_list: list, price_df: pd.DataFrame) -> pd.DataFrame:
    """
    Combines headlines matching the same day, embeds them, and creates a aligned
    dataframe matching the dates of our price data.
    """
    # Group news by date
    news_df = pd.DataFrame(news_list)

    if news_df.empty:
        # Fallback if no news was found: fill with zeros matching price dates
        print("No headlines found to embed. Generating baseline dummy embeddings.")
        embedding_dim = 768
        empty_embeddings = [np.zeros(embedding_dim) for _ in range(len(price_df))]
        emb_cols = [f"emb_{i}" for i in range(embedding_dim)]
        emb_df = pd.DataFrame(empty_embeddings, columns=emb_cols)
        return pd.concat([price_df.reset_index(drop=True), emb_df], axis=1)

    # Join headlines on the same date with a period
    grouped_news = news_df.groupby('date')['headline'].apply(lambda x: " . ".join(x)).reset_index()
    grouped_news['date'] = pd.to_datetime(grouped_news['date'])

    print("Generating embeddings for text data on GPU...")
    grouped_news['embedding'] = grouped_news['headline'].apply(get_text_embedding)

    # Expand the embedding array into individual columns
    embedding_dim = 768
    emb_cols = [f"emb_{i}" for i in range(embedding_dim)]
    emb_features = pd.DataFrame(grouped_news['embedding'].tolist(), columns=emb_cols)
    grouped_news = pd.concat([grouped_news.drop(columns=['embedding']), emb_features], axis=1)

    # Merge price data with news data on date
    price_df['date'] = pd.to_datetime(price_df['date'])
    merged_df = pd.merge(price_df, grouped_news, on='date', how='left')

    # For dates with no news headlines, fill embedding columns with 0
    merged_df[emb_cols] = merged_df[emb_cols].fillna(0)
    merged_df['headline'] = merged_df['headline'].fillna("No news available.")

    print(f"Data preprocessing complete. Final dataset shape: {merged_df.shape}")
    return merged_df

Using device: cuda for text embedding.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

# ----------------------------------------------------
# PART 6 & 7: MODEL BUILDING, TRAINING & EVALUATION
# ----------------------------------------------------

def train_multimodal_model(merged_df: pd.DataFrame):
    """
    Splits the data, trains a Random Forest on both price and text features,
    and returns the trained model alongside evaluation metrics.
    """
    print("Preparing features for model training...")

    # Define our numerical feature columns
    numerical_features = ['open', 'high', 'low', 'close', 'volume', 'sma_5', 'sma_20', 'volatility_5']

    # Define our embedding feature columns (all 768 dimensions)
    embedding_features = [f"emb_{i}" for i in range(768)]

    # Combine them into our final X feature matrix
    feature_cols = numerical_features + embedding_features

    X = merged_df[feature_cols]
    y = merged_df['target_next_close']

    # Split chronologically (Time-series data shouldn't be split randomly to avoid data leakage)
    # We will use the last 20% of data for testing
    split_idx = int(len(df) * 0.8) if 'df' in globals() else int(len(X) * 0.8)

    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    print(f"Training set size: {X_train.shape[0]} days")
    print(f"Testing set size: {X_test.shape[0]} days")

    # Initialize and train the model
    print("Training Random Forest Regressor (this may take a moment)...")
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

    # Predict and Evaluate
    predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, predictions)

    print("\n--- Model Evaluation Metrics ---")
    print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
    print(f"R-squared Score (R2): {r2:.4f}")
    print("--------------------------------")

    # Quick plot to visualize actual vs predicted prices in Colab
    plt.figure(figsize=(10, 5))
    plt.plot(y_test.values, label="Actual Next Close", color="blue", alpha=0.7)
    plt.plot(predictions, label="Predicted Next Close", color="orange", linestyle="--")
    plt.title("Commodity Price Prediction - Test Set Validation")
    plt.xlabel("Days")
    plt.ylabel("Price")
    plt.legend()
    plt.show()

    return model, feature_cols

In [7]:
import gradio as gr
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# ----------------------------------------------------
# PART 8: CORE BACKEND INFERENCE FUNCTION
# ----------------------------------------------------

def run_multimodal_pipeline(commodity_choice, start_date, end_date):
    """
    Orchestrates the entire process: pulls data, engineers features,
    fetches news, builds embeddings, trains the model, and outputs UI components.
    """
    # Map friendly UI names to Yahoo Finance tickers
    ticker_map = {
        "Gold": "GC=F",
        "Crude Oil": "CL=F",
        "Silver": "SI=F"
    }
    ticker = ticker_map[commodity_choice]

    try:
        # 1. Fetch market data (Part 2)
        price_df = fetch_commodity_data(ticker, start_date, end_date)

        # 2. Engineer technical indicators (Part 4)
        price_df = engineer_features(price_df)

        # 3. Fetch live news text (Part 3)
        news_list = fetch_commodity_news(commodity_choice)

        # 4. Process and embed text via GPU (Part 5)
        merged_df = aggregate_daily_news(news_list, price_df)

        # 5. Train model and evaluate (Part 6 & 7)
        model, feature_cols = train_multimodal_model(merged_df)

        # 6. Generate a final prediction for the latest day
        latest_row = merged_df.iloc[[-1]]
        latest_features = latest_row[feature_cols]
        prediction = model.predict(latest_features)[0]
        current_close = latest_row['close'].values[0]

        # Clean up directional signal text
        direction = "🚀 UPWARD TREND" if prediction > current_close else "📉 DOWNWARD TREND"
        change_pct = ((prediction - current_close) / current_close) * 100

        # 7. Create a high-quality visualization for the Gradio UI
        plt.figure(figsize=(10, 4.5))
        plt.plot(merged_df['date'], merged_df['close'], label='Historical Close', color='#1f77b4', linewidth=2)

        # Plot predicted point next to it
        tomorrow_date = merged_df['date'].iloc[-1] + timedelta(days=1)
        plt.scatter(tomorrow_date, prediction, color='#ff7f0e', s=100, label='Predicted Next Close', zorder=5)

        plt.title(f"{commodity_choice} Price History & Next-Day Prediction", fontsize=12, fontweight='bold')
        plt.xlabel("Date")
        plt.ylabel("Price (USD)")
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.legend(loc='upper left')
        plt.tight_layout()

        # Save plot to pass to Gradio
        plot_output = plt.gcf()

        # Formulate textual output summary
        summary_text = (
            f"### Analysis Summary for {commodity_choice}\n"
            f"- **Most Recent Close Price:** ${current_close:,.2f}\n"
            f"- **Predicted Next Close Price:** ${prediction:,.2f}\n"
            f"- **Expected Movement:** {direction} ({change_pct:+.2f}%)\n\n"
            f"**Latest Scraped Headline Scanned:**\n"
            f"\"{merged_df['headline'].iloc[-1][:120]}...\""
        )

        return plot_output, summary_text

    except Exception as e:
        # Error handling that reports neatly back to the UI interface
        fig, ax = plt.subplots(figsize=(5, 2))
        ax.text(0.5, 0.5, f"Error pipeline initialization failed:\n{str(e)}",
                ha='center', va='center', color='red', fontsize=10)
        ax.axis('off')
        return fig, f"### Pipeline Interrupted\nAn error occurred while loading data: {e}"




In [8]:
# ----------------------------------------------------
# PART 9: DESIGNING THE GRADIO UI
# ----------------------------------------------------

# Custom CSS theme for a sleek look
custom_theme = gr.themes.Default(
    primary_hue="amber",
    secondary_hue="slate",
).set(
    body_background_fill="*neutral_50",
    block_background_fill="white",
    block_border_width="1px"
)

with gr.Blocks(theme=custom_theme, title="Multimodal Commodity Dashboard") as demo:
    gr.Markdown(
        """
        # 📊 Multimodal Commodity Prediction Dashboard
        Predicting next-day trends by combining **technical financial indicators** with **live text sentiment analysis** parsed via a T4 GPU Transformer model.
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Input Parameters")
            commodity_input = gr.Dropdown(
                choices=["Gold", "Crude Oil", "Silver"],
                value="Gold",
                label="Select Commodity Asset"
            )
            start_date_input = gr.Textbox(
                value="2025-01-01",
                label="Historical Start Date (YYYY-MM-DD)"
            )
            end_date_input = gr.Textbox(
                value="2026-01-01",
                label="Historical End Date (YYYY-MM-DD)"
            )
            submit_btn = gr.Button("🚀 Run Multimodal Forecast", variant="primary")

        with gr.Column(scale=2):
            gr.Markdown("### AI Predictive Engine Output")
            output_markdown = gr.Markdown("Select options and press execute to view insights.")
            output_plot = gr.Plot(label="Market History & Forecast Plot")

    # Set up button action event listeners
    submit_btn.click(
        fn=run_multimodal_pipeline,
        inputs=[commodity_input, start_date_input, end_date_input],
        outputs=[output_plot, output_markdown]
    )

# Note: Part 10 (Launching) happens right below!

/tmp/ipykernel_22944/418409442.py:15: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=custom_theme, title="Multimodal Commodity Dashboard") as demo:


In [15]:
#corrections and misses



import requests
from bs4 import BeautifulSoup
from datetime import datetime

def fetch_commodity_news(commodity_name: str) -> list:
    print(f"Fetching fresh macro news headlines for: {commodity_name}...")

    # Updated to a more reliable, high-volume financial RSS endpoint
    url = "https://www.yahoo.com/news/rss/finance"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    # Expanded keyword matching so Gold tracks massive macro movers too
    aliases = {
        "Gold": ["gold", "precious metal", "bullion", "fed", "inflation", "rate cut", "commodities"],
        "Crude Oil": ["oil", "crude", "wti", "brent", "energy", "opec", "gasoline"],
        "Silver": ["silver", "precious metal", "bullion", "commodities", "industrial metal"]
    }

    keywords = aliases.get(commodity_name, [commodity_name.lower()])

    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, features="xml")
        items = soup.find_all('item')

        news_data = []
        for item in items:
            title = item.title.text if item.title else ""
            pub_date_str = item.pubDate.text if item.pubDate else ""

            # Match titles against our broadened keyword group
            if any(k in title.lower() for k in keywords):
                try:
                    parsed_date = datetime.strptime(pub_date_str[:25].strip(), "%a, %d %b %Y %H:%M:%S")
                    formatted_date = parsed_date.strftime("%Y-%m-%d")
                except Exception:
                    formatted_date = datetime.today().strftime("%Y-%m-%d")

                news_data.append({
                    "date": formatted_date,
                    "headline": title
                })

        print(f"📊 Pipeline Match: Found {len(news_data)} macro headlines for '{commodity_name}'.")
        return news_data

    except Exception as e:
        print(f"Bypassing text feed exception: {e}")
        return []

# Re-launch the demo now that the function is explicitly available


In [10]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
from datetime import datetime

# ------------------------------------------------------------------
# FIXED PART 2: BULLETPROOF DATA FETCHER
# ------------------------------------------------------------------
def fetch_commodity_data(ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
    print(f"Fetching market data for {ticker}...")
    df = yf.download(ticker, start=start_date, end=end_date)

    if df.empty:
        raise ValueError(f"No data found for ticker {ticker}. Verify network or date configuration.")

    # Secure fix for multi-index columns often thrown by yfinance on Gold/Oil
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.reset_index()
    df.columns = [str(col).lower().replace(' ', '_') for col in df.columns]

    # Calculate target label
    df['target_next_close'] = df['close'].shift(-1)
    df = df.dropna(subset=['target_next_close'])

    print(f"Successfully loaded {len(df)} price vectors.")
    return df

# ------------------------------------------------------------------
# FIXED PART 3: BROADENED RSS HEADLINE ALIGNMENT
# ------------------------------------------------------------------
def fetch_commodity_news(commodity_name: str) -> list:
    print(f"Fetching global news headlines relevant to: {commodity_name}...")
    url = "https://finance.yahoo.com/news/rss"
    headers = {"User-Agent": "Mozilla/5.0"}

    # Flexible keyword aliases to catch more relevant feed articles
    aliases = {
        "Gold": ["gold", "precious metal", "bullion", "cmx", "commodity"],
        "Crude Oil": ["oil", "crude", "wti", "brent", "energy", "commodity"],
        "Silver": ["silver", "precious metal", "bullion", "commodity"]
    }

    keywords = aliases.get(commodity_name, [commodity_name.lower()])

    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, features="xml")
        items = soup.find_all('item')

        news_data = []
        for item in items:
            title = item.title.text if item.title else ""
            pub_date_str = item.pubDate.text if item.pubDate else ""

            # Match if any alias keyword hits the feed headline text
            if any(k in title.lower() for k in keywords):
                try:
                    parsed_date = datetime.strptime(pub_date_str[:25].strip(), "%a, %d %b %Y %H:%M:%S")
                    formatted_date = parsed_date.strftime("%Y-%m-%d")
                except Exception:
                    formatted_date = datetime.today().strftime("%Y-%m-%d")

                news_data.append({
                    "date": formatted_date,
                    "headline": title
                })

        print(f"Acquired {len(news_data)} headlines for analysis matrix.")
        return news_data
    except Exception as e:
        print(f"News fetch bypass notice: {e}")
        return []


In [11]:
# ------------------------------------------------------------------
# ROBUST ORCHESTRATOR WITH AUTOMATIC GOLD FALLBACK
# ------------------------------------------------------------------

def run_multimodal_pipeline(commodity_choice, start_date, end_date):
    """
    Orchestrates data fetching, feature engineering, text analysis,
    and prediction modeling with a built-in safety fallback for Gold data structures.
    """
    # Primary ticker mappings
    ticker_map = {
        "Gold": "GC=F",
        "Crude Oil": "CL=F",
        "Silver": "SI=F"
    }
    ticker = ticker_map[commodity_choice]

    try:
        # 1. Fetch market data (With smart fallback processing)
        try:
            price_df = fetch_commodity_data(ticker, start_date, end_date)
        except Exception as primary_err:
            if commodity_choice == "Gold":
                print(f"Primary symbol GC=F failed. Activating high-reliability fallback 'GLD'...")
                price_df = fetch_commodity_data("GLD", start_date, end_date)
            else:
                raise primary_err

        # 2. Engineer technical indicators
        price_df = engineer_features(price_df)

        # 3. Fetch live news headlines
        news_list = fetch_commodity_news(commodity_choice)

        # 4. Extract text vectors using the T4 GPU
        merged_df = aggregate_daily_news(news_list, price_df)

        # 5. Train and validate our machine learning system
        model, feature_cols = train_multimodal_model(merged_df)

        # 6. Generate the predictive inference metrics
        latest_row = merged_df.iloc[[-1]]
        latest_features = latest_row[feature_cols]
        prediction = model.predict(latest_features)[0]
        current_close = latest_row['close'].values[0]

        direction = "🚀 UPWARD TREND" if prediction > current_close else "📉 DOWNWARD TREND"
        change_pct = ((prediction - current_close) / current_close) * 100

        # 7. Render custom visual output interface charts
        plt.figure(figsize=(10, 4.5))
        plt.plot(merged_df['date'], merged_df['close'], label='Historical Close', color='#1f77b4', linewidth=2)

        tomorrow_date = merged_df['date'].iloc[-1] + timedelta(days=1)
        plt.scatter(tomorrow_date, prediction, color='#ff7f0e', s=100, label='Predicted Next Close', zorder=5)

        plt.title(f"{commodity_choice} Price History & Next-Day Prediction", fontsize=12, fontweight='bold')
        plt.xlabel("Date")
        plt.ylabel("Price (USD)")
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.legend(loc='upper left')
        plt.tight_layout()

        plot_output = plt.gcf()

        summary_text = (
            f"### Analysis Summary for {commodity_choice}\n"
            f"- **Most Recent Close Price:** ${current_close:,.2f}\n"
            f"- **Predicted Next Close Price:** ${prediction:,.2f}\n"
            f"- **Expected Movement:** {direction} ({change_pct:+.2f}%)\n\n"
            f"**Latest Scraped Headline Scanned:**\n"
            f"\"{merged_df['headline'].iloc[-1][:120]}...\""
        )

        return plot_output, summary_text

    except Exception as e:
        fig, ax = plt.subplots(figsize=(5, 2))
        ax.text(0.5, 0.5, f"Pipeline Error:\n{str(e)}", ha='center', va='center', color='red', fontsize=10)
        ax.axis('off')
        return fig, f"### Pipeline Interrupted\nAn error occurred while handling asset parameters: {e}"



In [12]:
##solving pipeline error for gold
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

def run_multimodal_pipeline(commodity_choice, start_date, end_date):
    """
    Orchestrates the data pipeline, model training, and prediction visualization.
    Includes comprehensive fail-safes for missing data or text attributes.
    """
    ticker_map = {
        "Gold": "GC=F",
        "Crude Oil": "CL=F",
        "Silver": "SI=F"
    }
    ticker = ticker_map[commodity_choice]

    try:
        # 1. Fetch market data with smart fallback tracking
        try:
            price_df = fetch_commodity_data(ticker, start_date, end_date)
        except Exception as primary_err:
            if commodity_choice == "Gold":
                print("Primary symbol GC=F failed. Activating high-reliability fallback 'GLD'...")
                price_df = fetch_commodity_data("GLD", start_date, end_date)
            else:
                raise primary_err

        # 2. Engineer technical indicators
        price_df = engineer_features(price_df)

        # 3. Fetch live news headlines
        news_list = fetch_commodity_news(commodity_choice)

        # 4. Extract text vectors using the T4 GPU
        merged_df = aggregate_daily_news(news_list, price_df)

        # 5. Train and validate our machine learning system
        model, feature_cols = train_multimodal_model(merged_df)

        # 6. Generate the predictive inference metrics
        latest_row = merged_df.iloc[[-1]]
        latest_features = latest_row[feature_cols]
        prediction = model.predict(latest_features)[0]
        current_close = latest_row['close'].values[0]

        direction = "🚀 UPWARD TREND" if prediction > current_close else "📉 DOWNWARD TREND"
        change_pct = ((prediction - current_close) / current_close) * 100

        # Safe check for headline availability to prevent KeyErrors
        latest_headline = "No current headlines found matching this asset."
        if 'headline' in merged_df.columns:
            latest_headline = str(merged_df['headline'].iloc[-1])

        # 7. Render visual output charts
        plt.figure(figsize=(10, 4.5))
        plt.plot(merged_df['date'], merged_df['close'], label='Historical Close', color='#1f77b4', linewidth=2)

        tomorrow_date = merged_df['date'].iloc[-1] + timedelta(days=1)
        plt.scatter(tomorrow_date, prediction, color='#ff7f0e', s=100, label='Predicted Next Close', zorder=5)

        plt.title(f"{commodity_choice} Price History & Next-Day Prediction", fontsize=12, fontweight='bold')
        plt.xlabel("Date")
        plt.ylabel("Price (USD)")
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.legend(loc='upper left')
        plt.tight_layout()

        plot_output = plt.gcf()

        summary_text = (
            f"### Analysis Summary for {commodity_choice}\n"
            f"- **Most Recent Close Price:** ${current_close:,.2f}\n"
            f"- **Predicted Next Close Price:** ${prediction:,.2f}\n"
            f"- **Expected Movement:** {direction} ({change_pct:+.2f}%)\n\n"
            f"**Latest Scraped Headline Scanned:**\n"
            f"\"{latest_headline[:120]}...\""
        )

        return plot_output, summary_text

    except Exception as e:
        import traceback
        traceback.print_exc() # Prints detailed debug logs to your Colab console window
        fig, ax = plt.subplots(figsize=(5, 2))
        ax.text(0.5, 0.5, f"Pipeline Error:\n{str(e)}",ha='center', va='center', color='red', fontsize=10)
        ax.axis('off')
        return fig, f"### Pipeline Interrupted\nAn error occurred while handling asset parameters: {e}"



In [13]:
import numpy as np
import matplotlib.pyplot as plt
import gradio as gr
from datetime import datetime, timedelta

# ----------------------------------------------------
# 1. BULLETPROOF INFERENCE ENGINE (WITH HEADLINE SAFETY)
# ----------------------------------------------------
def run_multimodal_pipeline(commodity_choice, start_date, end_date):
    ticker_map = {
        "Gold": "GC=F",
        "Crude Oil": "CL=F",
        "Silver": "SI=F"
    }
    ticker = ticker_map[commodity_choice]

    try:
        # Fetch market data with smart fallback tracking
        try:
            price_df = fetch_commodity_data(ticker, start_date, end_date)
        except Exception as primary_err:
            if commodity_choice == "Gold":
                print("Primary symbol GC=F failed. Activating high-reliability fallback 'GLD'...")
                price_df = fetch_commodity_data("GLD", start_date, end_date)
            else:
                raise primary_err

        # Engineer technical indicators
        price_df = engineer_features(price_df)

        # Fetch live news headlines
        news_list = fetch_commodity_news(commodity_choice)

        # Extract text vectors using the T4 GPU
        merged_df = aggregate_daily_news(news_list, price_df)

        # Train and validate our machine learning system
        model, feature_cols = train_multimodal_model(merged_df)

        # Generate the predictive inference metrics
        latest_row = merged_df.iloc[[-1]]
        latest_features = latest_row[feature_cols]
        prediction = model.predict(latest_features)[0]
        current_close = latest_row['close'].values[0]

        direction = "🚀 UPWARD TREND" if prediction > current_close else "📉 DOWNWARD TREND"
        change_pct = ((prediction - current_close) / current_close) * 100

        # EXPLICIT FAIL-SAFE: Check if headline column exists before indexing
        if 'headline' in merged_df.columns and not merged_df['headline'].empty:
            latest_headline = str(merged_df['headline'].iloc[-1])
        else:
            latest_headline = "No active news articles scanned for this token today."

        # Render visual output charts
        plt.figure(figsize=(10, 4.5))
        plt.plot(merged_df['date'], merged_df['close'], label='Historical Close', color='#1f77b4', linewidth=2)

        tomorrow_date = merged_df['date'].iloc[-1] + timedelta(days=1)
        plt.scatter(tomorrow_date, prediction, color='#ff7f0e', s=100, label='Predicted Next Close', zorder=5)

        plt.title(f"{commodity_choice} Price History & Next-Day Prediction", fontsize=12, fontweight='bold')
        plt.xlabel("Date")
        plt.ylabel("Price (USD)")
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.legend(loc='upper left')
        plt.tight_layout()

        plot_output = plt.gcf()

        summary_text = (
            f"### Analysis Summary for {commodity_choice}\n"
            f"- **Most Recent Close Price:** ${current_close:,.2f}\n"
            f"- **Predicted Next Close Price:** ${prediction:,.2f}\n"
            f"- **Expected Movement:** {direction} ({change_pct:+.2f}%)\n\n"
            f"**Latest Scraped Headline Scanned:**\n"
            f"\"{latest_headline[:120]}...\""
        )

        return plot_output, summary_text

    except Exception as e:
        fig, ax = plt.subplots(figsize=(5, 2))
        ax.text(0.5, 0.5, f"Pipeline Error:\n{str(e)}", ha='center', va='center', color='red', fontsize=10)
        ax.axis('off')
        return fig, f"### Pipeline Interrupted\nAn error occurred while handling asset parameters: {e}"


# ----------------------------------------------------
# 2. REBUILD GRAPHICAL INTERFACE LAYER
# ----------------------------------------------------
custom_theme = gr.themes.Default(primary_hue="amber", secondary_hue="slate")

with gr.Blocks(theme=custom_theme, title="Multimodal Commodity Dashboard") as new_demo:
    gr.Markdown(
        """
        # 📊 Multimodal Commodity Prediction Dashboard
        Predicting next-day trends by combining **technical financial indicators** with **live text sentiment analysis** parsed via a T4 GPU Transformer model.
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Input Parameters")
            commodity_input = gr.Dropdown(choices=["Gold", "Crude Oil", "Silver"], value="Gold", label="Select Commodity Asset")
            start_date_input = gr.Textbox(value="2025-01-01", label="Historical Start Date (YYYY-MM-DD)")
            end_date_input = gr.Textbox(value="2026-01-01", label="Historical End Date (YYYY-MM-DD)")
            submit_btn = gr.Button("🚀 Run Multimodal Forecast", variant="primary")

        with gr.Column(scale=2):
            gr.Markdown("### AI Predictive Engine Output")
            output_markdown = gr.Markdown("Select options and press execute to view insights.")
            output_plot = gr.Plot(label="Market History & Forecast Plot")

    submit_btn.click(
        fn=run_multimodal_pipeline,
        inputs=[commodity_input, start_date_input, end_date_input],
        outputs=[output_plot, output_markdown]
    )




/tmp/ipykernel_22944/970231158.py:94: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=custom_theme, title="Multimodal Commodity Dashboard") as new_demo:


In [16]:
import os
import gradio as gr
from google import genai
from google.colab import userdata

# 1. Initialize the official Google GenAI Client safely using Colab Secrets
try:
    api_key = userdata.get('GEMINI_API_KEY')
    ai_client = genai.Client(api_key=api_key)
except Exception:
    print("Warning: GEMINI_API_KEY not found in Colab Secrets. Chatbot will return placeholder text.")
    ai_client = None

# 2. Define the Chatbot Backend Function
def commodity_chatbot(message, history):
    """
    Handles user queries about commodities using the Gemini 2.5 Flash model.
    """
    if not ai_client:
        return "System Error: Please configure your GEMINI_API_KEY in Colab Secrets to activate the assistant."

    system_instruction = (
        "You are an expert financial commodities assistant. Provide concise, grounded insights "
        "regarding market trends, macroeconomic factors, and historical context for Gold, Crude Oil, and Silver."
    )

    try:
        response = ai_client.models.generate_content(
            model='gemini-2.5-flash',
            contents=message,
            config={'system_instruction': system_instruction}
        )
        return response.text
    except Exception as e:
        return f"Failed to reach Gemini engine: {str(e)}"


# ----------------------------------------------------
# 3. REBUILD THE TWO-TAB GRADIO DASHBOARD (UPDATED API)
# ----------------------------------------------------
# Removed 'theme' from here to comply with Gradio 6.0+ rules
with gr.Blocks(title="Multimodal Commodity Hub") as advanced_demo:
    gr.Markdown(
        """
        # 📊 Advanced Multimodal Commodity Hub
        Combine predictive machine learning forecasting with a live generative AI expert assistant.
        """
    )

    with gr.Tabs():
        # TAB 1: Our existing ML predictive system
        with gr.TabItem("📈 Predictive ML Engine"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### Input Parameters")
                    commodity_input = gr.Dropdown(choices=["Gold", "Crude Oil", "Silver"], value="Gold", label="Select Commodity Asset")
                    start_date_input = gr.Textbox(value="2025-01-01", label="Historical Start Date (YYYY-MM-DD)")
                    end_date_input = gr.Textbox(value="2026-01-01", label="Historical End Date (YYYY-MM-DD)")
                    submit_btn = gr.Button("🚀 Run Multimodal Forecast", variant="primary")

                with gr.Column(scale=2):
                    gr.Markdown("### AI Predictive Engine Output")
                    output_markdown = gr.Markdown("Select options and press execute to view insights.")
                    output_plot = gr.Plot(label="Market History & Forecast Plot")

            # Link the ML event listener
            submit_btn.click(
                fn=run_multimodal_pipeline,
                inputs=[commodity_input, start_date_input, end_date_input],
                outputs=[output_plot, output_markdown]
            )

        # TAB 2: The Gemini Chatbot interface
        with gr.TabItem("💬 Gemini Market Assistant"):
            gr.Markdown("### Ask Gemini about Gold, Crude Oil, or Silver trends, news, or macros:")

            # Fixed: Removed the unexpected 'type="messages"' argument
            gr.ChatInterface(
                fn=commodity_chatbot,
                examples=[
                    "What macroeconomic factors typically cause Gold prices to rally?",
                    "How does a strong US dollar affect Crude Oil prices?",
                    "What is the historical industrial relationship between Silver and Gold?"
                ]
            )

# ----------------------------------------------------
# 4. LAUNCH CLEANLY (PASSING THEME HERE)
# ----------------------------------------------------
custom_theme = gr.themes.Default(primary_hue="amber", secondary_hue="slate")

gr.close_all()
# Handled theme assignment directly inside launch parameters for Gradio 6 compliance
advanced_demo.launch(share=True, theme=custom_theme)

Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f826dbf32f52350784.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
